In [2]:
# Cell 1: Imports & Setup
import pandas as pd
import numpy as np
import torch
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

In [3]:
from datasets import Dataset as HFDataset
import evaluate

# HuggingFace 🤗 transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

In [5]:
#Detects whether a GPU (CUDA) or Apple’s MPS backend is available, and otherwise falls back to CPU.
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("→ Using device:", device)


→ Using device: mps


In [14]:
mind_news_path = Path("/Users/harshadayiniakula/Desktop/RS/newcat.csv")

# Load the TSV; columns as per MIND spec
mind_df = pd.read_csv(
    mind_news_path)

In [16]:
mind_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   newid              51282 non-null  object
 1   vertical           51282 non-null  object
 2   subvertical        51282 non-null  object
 3   title              51282 non-null  object
 4   abstract           48616 non-null  object
 5   url                51282 non-null  object
 6   title_entities     51279 non-null  object
 7   abstract_entities  51278 non-null  object
 8   new_category       51282 non-null  object
dtypes: object(9)
memory usage: 3.5+ MB


In [24]:
mind_df["text"] = mind_df["title"].fillna("") + " " + mind_df["abstract"].fillna("")


In [18]:
newcat = sorted(mind_df["new_category"].unique())  
newcat2id = {v: i for i, v in enumerate(newcat)} #maps subcategory to integers
id2newcat = {i: v for v, i in newcat2id.items()} #inverse mapping of vertical2id

mind_df["label"] = mind_df["new_category"].map(newcat2id) #adds label column by mapping each subcategory to its integer id


In [20]:
print(newcat)

['autos', 'entertainment', 'finance', 'foodanddrink', 'health', 'lifestyle', 'movies', 'music', 'news', 'newscrime', 'newspolitics', 'newsscienceandtechnology', 'newstrends', 'newsworld', 'sports', 'travel', 'tv', 'video', 'weather']


In [26]:
# Build a HuggingFace Dataset
hf_dataset = HFDataset.from_pandas(
    mind_df[["newid", "text", "label"]].rename(columns={"newid": "id"})
)
# Split 90% train / 10% validation
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]

print(f"Train size: {len(train_ds)}, Validation size: {len(val_ds)}")

Train size: 46153, Validation size: 5129


In [32]:
MODEL_NAME = "xlm-roberta-base" #basic multilingual tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256, #truncates after 256 length 
    #Returns the usual input_ids and attention_mask.
)
#Returns the usual input_ids and attention_mask.

In [34]:
train_tok = train_ds.map(tokenize, batched=True)
val_tok   = val_ds.map(tokenize,   batched=True)

# We only need input_ids & attention_mask; rename label to labels for Trainer
train_tok = train_tok.rename_column("label", "labels")
val_tok   = val_tok.rename_column("label",   "labels")
train_tok.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_tok.set_format(  "torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/46153 [00:00<?, ? examples/s]

Map:   0%|          | 0/5129 [00:00<?, ? examples/s]

In [35]:
# Cell 4: Metrics, model, Trainer setup
# Load metrics via evaluate
accuracy = evaluate.load("accuracy")
f1_score = evaluate.load("f1")


In [38]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_score.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

#Defines a helper that the Trainer will call after each evaluation:
# Takes raw logits and true labels.
# Computes predicted classes via argmax.
# Returns a dict of metric values.

In [40]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(newcat)
).to(device)

data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir="telugu_classifier",
    eval_strategy="epoch",        # updated from deprecated `evaluation_strategy`
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_23348/1806429019.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [42]:
# Cell 5: Train & evaluate
trainer.train()
metrics = trainer.evaluate()
print(metrics)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.942100,0.930637,0.702866,0.603757
2,0.786800,0.845085,0.736011,0.662902
3,0.670100,0.831269,0.738546,0.666042


{'eval_loss': 0.8312690854072571, 'eval_accuracy': 0.7385455254435562, 'eval_f1_macro': 0.6660423233941491, 'eval_runtime': 329.2573, 'eval_samples_per_second': 15.577, 'eval_steps_per_second': 0.489, 'epoch': 3.0}


zsh:1: no matches found: /Users/harshadayiniakula/.cache/huggingface/transformers/*


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [60]:
# Save model and tokenizer after training
model.save_pretrained("telugu_classifier_newcat")
tokenizer.save_pretrained("telugu_classifier_newcat")


('telugu_classifier_newcat/tokenizer_config.json',
 'telugu_classifier_newcat/special_tokens_map.json',
 'telugu_classifier_newcat/tokenizer.json')

In [ ]:
# Cell X: Load your fine-tuned Telugu classifier
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# — this is the folder you saved to in Cell 13 —
MODEL_DIR = "telugu_classifier_newcat"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# move model to the same device you set earlier
model.to(device)

print(f"Loaded model & tokenizer from {MODEL_DIR} on {device}")


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("telugu_classifier_newcat").to(device)
tokenizer = AutoTokenizer.from_pretrained("telugu_classifier_newcat")


In [ ]:
# Cell 6: Load & concatenate ALL Telugu articles from Parquet
telugu_dir = Path("/Users/harshadayiniakula/Desktop/RS/telugu")
parquets = list(telugu_dir.glob("*.parquet"))

# Reload + dedupe + re-index
telugu_df = (
    pd.concat([pd.read_parquet(p) for p in parquets], ignore_index=True)
      .drop_duplicates(subset="story_id")
      .reset_index(drop=True)       # ← reset so indices run 0…N-1
)

telugu_df["text"] = (
    telugu_df["headline"].fillna("") + " " + telugu_df["article"].fillna("")
)

print("After dedupe & reset:", len(telugu_df), "rows, indices 0…", telugu_df.index[-1])



In [ ]:
# Save telugu_df for future quick loads
telugu_df.to_pickle("telugu_df_full.pkl")
print("Saved telugu_df_full.pkl with", len(telugu_df), "rows")


In [62]:
import pandas as pd

telugu_df = pd.read_pickle("telugu_df_full.pkl")
print("Loaded telugu_df with", len(telugu_df), "rows")

Loaded telugu_df with 46816 rows


In [ ]:
I ran until here only, cell 6 shouldnt have been running again unnecessarily, i will remove it if i can

In [66]:
batch_size = 64  


In [68]:
# Cell 7a: Try loading cached probs first
import os, numpy as np, pandas as pd

if os.path.exists("telugu_all_probs_newcat.npy"):
    print("Loading cached all_probs…")
    all_probs = np.load("telugu_all_probs_newcat.npy")
    # reload telugu_df if needed to get back story_id alignment
    telugu_df = pd.read_csv("telugu_story_ids_newcat.csv").merge(
        telugu_df, on="story_id", how="left"
    )
    print("Loaded all_probs shape:", all_probs.shape)
else:
    # … your existing batch‐loop to compute all_probs …
    all_probs = []
    model.eval()
    for i in range(0, len(telugu_df), batch_size):
        batch_texts = telugu_df["text"].iloc[i : i + batch_size].tolist()
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()

        all_probs.append(probs)

    all_probs = np.vstack(all_probs)
    
    # then cache:
    np.save("telugu_all_probs_newcat.npy", all_probs)
    telugu_df[["story_id"]].to_csv("telugu_story_ids_newcat.csv", index=False)

print("all_probs ready; shape =", all_probs.shape)


all_probs ready; shape = (46816, 19)


In [69]:
# Cell 7b: Cache all_probs to disk
import numpy as np

# Save the raw array
np.save("telugu_all_probs_newcat.npy", all_probs)

# (Optional) also save the corresponding story IDs so you can re‐zip later
import pandas as pd
telugu_df[["story_id"]].to_csv("telugu_story_ids_newcat.csv", index=False)

print("✅ Cached all_probs (shape", all_probs.shape, ") and story_ids")


✅ Cached all_probs (shape (46816, 19) ) and story_ids


In [72]:
# Cell 7c: Build item_feature_tuples_telugu
item_feature_tuples_telugu = [
    (
        str(story_id),
        {
            id2newcat[i]: float(prob_vec[i])
            for i in range(len(newcat))
            if prob_vec[i] > 1e-6
        }
    )
    for story_id, prob_vec in zip(telugu_df["story_id"], all_probs)
]

print(f"Built {len(item_feature_tuples_telugu)} feature‐tuples")


Built 46816 feature‐tuples


In [76]:
from lightfm.data import Dataset as LFDataset


/opt/anaconda3/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [80]:
telugu_ids = [str(s) for s in telugu_df["story_id"]]


In [86]:
from lightfm.data import Dataset

telugu_ids = [str(s) for s in telugu_df["story_id"]]
newcat  = sorted(mind_df["new_category"].unique())


telugu_ds = Dataset(
    user_identity_features=False,
    item_identity_features=False
)
telugu_ds.fit(
    users=[],                 # no user metadata
    items=telugu_ids,         # <-- your 46 816 article IDs
    user_features=[],         # none
    item_features=newcat   # <-- exactly the 17 vertical labels
)


# 1) Build the manual mappings
telugu_ids      = [str(s) for s in telugu_df["story_id"]]
item_map        = {sid: idx for idx, sid in enumerate(telugu_ids)}
feature_map     = {v: i   for i, v in enumerate(newcat)}

In [82]:
lf_dataset = LFDataset(
    user_identity_features=False,
    item_identity_features=False
)
lf_dataset.fit(
    users=[],
    items=telugu_ids,        # your 46 816 story IDs
    user_features=[],        # none
    item_features=newcat  # exactly the 17 vertical names
)


In [88]:

# 2) Assemble the sparse matrix data
rows, cols, data = [], [], []
for sid, probs in zip(telugu_ids, all_probs):
    i = item_map[sid]
    for v, p in zip(newcat, probs):
        if p > 1e-6:
            rows.append(i)
            cols.append(feature_map[v])
            data.append(p)

# 3) Create the CSR matrix
item_features_telugu = csr_matrix(
    (data, (rows, cols)),
    shape=(len(telugu_ids), len(newcat))
)

# 4) Verify
print("Manual item_map size   :", len(item_map))        # → 46816
print("Manual feature_map size:", len(feature_map))     # → 17
print("CSR shape:", item_features_telugu.shape)         # → (46816, 17)

Manual item_map size   : 46816
Manual feature_map size: 19
CSR shape: (46816, 19)


In [90]:
# Cell DebugManualMap: Verify manual item_map, feature_map, and CSR matrix

import numpy as np
import random

# Assume these exist:
# telugu_ids         -> list of story_id strings
# verticals          -> list of 17 vertical names, in the same order as feature_map
# all_probs          -> np.ndarray of shape (n_items, n_features)
# item_map           -> dict mapping story_id -> row index
# feature_map        -> dict mapping vertical -> col index
# item_features_telugu -> csr_matrix of shape (n_items, n_features)

n_items = len(telugu_ids)
n_feats = len(newcat)

# 1) Check keys and values in item_map
assert set(item_map.keys()) == set(telugu_ids), "Item IDs mismatch!"
assert set(item_map.values()) == set(range(n_items)), "Item indices not 0..n-1!"

# 2) Check keys and values in feature_map
assert set(feature_map.keys()) == set(newcat), "Feature names mismatch!"
assert set(feature_map.values()) == set(range(n_feats)), "Feature indices not 0..n_feats-1!"

# 3) Check CSR shape
assert item_features_telugu.shape == (n_items, n_feats), "CSR matrix has wrong shape!"

# 4) Check that no row is empty (each article has ≥1 feature)
row_sums = item_features_telugu.sum(axis=1).A1
assert np.all(row_sums > 0), "Some rows have zero total weight!"

# 5) Check that CSR entries match thresholded all_probs
threshold = 1e-6
# Build dense thresholded array
expected = all_probs * (all_probs > threshold)
# Compare
dense_manual = item_features_telugu.toarray()
assert np.allclose(dense_manual, expected), "CSR values do not match all_probs!"

# 6) Spot‐check a few random articles
for sid in random.sample(telugu_ids, 5):
    row_idx = item_map[sid]
    # Get nonzero cols & values
    nz_cols = item_features_telugu[row_idx].tocoo().col
    nz_vals = item_features_telugu[row_idx].tocoo().data
    # Compare to all_probs
    probs = all_probs[row_idx]
    for col, val in zip(nz_cols, nz_vals):
        assert abs(probs[col] - val) < 1e-6, f"Mismatch at item {sid}, feat {verticals[col]}"

print("✅ Manual mapping and CSR matrix are fully verified and aligned.")


✅ Manual mapping and CSR matrix are fully verified and aligned.


In [92]:
# Cell: Generate Top-K Telugu recommendations for English users

import numpy as np
import pandas as pd
import pickle
from lightfm import LightFM

# 1) Load your pretrained English LightFM model
#    (Assuming you saved it after training, e.g. via pickle)
with open("lightfm_model_newcat.pkl", "rb") as f:
    model: LightFM = pickle.load(f)

# 2) Prepare the list of Telugu item indices and their string IDs
#    (this must match the order you used in your manual mapping)
telugu_ids = [str(s) for s in telugu_df["story_id"]]
item_idx  = np.array([ item_map[sid] for sid in telugu_ids ])

# 3) For each user, predict scores for all Telugu items
#    Then take Top-K. Example for user 123 (internal ID u_idx):
u_idx = 0  # internal user index, 0-based in your train_interactions matrix
scores = model.predict(
    user_ids=u_idx,
    item_ids=item_idx,
    item_features=item_features_telugu
)

# 4) Extract Top-K (e.g. K=10)
K = 10
top_k_indices = np.argsort(-scores)[:K]
top_k_story_ids = [ telugu_ids[i] for i in top_k_indices ]
top_k_scores    = scores[top_k_indices]

recommendations = pd.DataFrame({
    "story_id": top_k_story_ids,
    "score":    top_k_scores
})
print("Top-10 Telugu recs for user", u_idx)
print(recommendations)

# 5) Batch-compute for all users in your test set
from lightfm.evaluation import auc_score

# Suppose test_users is a list of internal user indices you want to evaluate.
test_users = [0,1,2,3,4]  # example
all_recs = {}
for u in test_users:
    s = model.predict(u, item_idx, item_features=item_features_telugu)
    topk = np.argsort(-s)[:K]
    all_recs[u] = [ telugu_ids[i] for i in topk ]

# 6) (Optional) Evaluate AUC on a held-out English×Telugu interactions matrix
#    If you have any real interactions to compare against:
# auc = auc_score(model, test_interactions_telugu, item_features=item_features_telugu).mean()
# print("Telugu AUC:", auc)


Top-10 Telugu recs for user 0
  story_id     score
0  2534930  7.036376
1  2544223  6.756736
2  2544282  6.756736
3  2516579  6.617276
4  2483635  6.607616
5  2483693  6.607616
6  2483620  6.607616
7  2532131  6.573864
8  2532499  6.573864
9  2532309  6.573864


In [94]:
# Cell: Inspect Telugu article → vertical assignments
import numpy as np

def print_telugu_assignment(idx, top_k=3):
    """
    Print headline, article text, and top_k predicted verticals for the
    Telugu article at DataFrame index `idx`.
    """
    row = telugu_df.iloc[idx]
    print(f"Story ID : {row['story_id']}")
    print(f"Headline : {row['headline']}")
    print(f"Article  :\n{row['article']}\n")
    
    probs   = all_probs[idx]                     # shape (17,)
    top_idxs = np.argsort(-probs)[:top_k]        # indices of top_k verticals
    
    print("Top vertical assignments:")
    for rank, i in enumerate(top_idxs, start=1):
        print(f"  {rank}. {newcat[i]}  (p = {probs[i]:.3f})")
    print("\n" + "-"*80 + "\n")

# Example: inspect the first 5 articles
for idx in range(5):
    print_telugu_assignment(idx, top_k=3)


Story ID : 920427
Headline : శేఖర్ కమ్ముల తండ్రి శేషయ్య కన్నుమూత
Article  :
ప్రముఖ దర్శకులు శేఖర్ కమ్ముల ఇంట విషాదం నెలకొంది . ఆయన తండ్రి కమ్ముల శేషయ్య ( 89 ) కన్నుమూశారు . ఆయన గత కొద్దికాలంగా వృద్దాప్య సంబంధింత సమస్యలతో బాధపడుతున్నారు . ప్రస్తుత పరిస్థితుల్లో కొద్దికాలంగా ఇంటి వద్దే చికిత్స అందిస్తున్నారు . ఇటీవల ఆరోగ్యం క్షీణించడంతో స్థానిక ప్రైవేట్ హాస్పిటల్లో చేర్పించారు . అక్కడే చికిత్స పొందుతూ శనివారం ఉదయం 6 గంటలకు మరణించారు అని సన్నిహితులు తెలిపారు . శనివారం సాయంత్రం బన్సీలాల్ పేట స్మశాన వాటికలో అంత్యక్రియలు జరుగుతాయని కుటుంబ సభ్యులు తెలిపారు . శేఖర్ కమ్ముల లాక్డౌన్ సమయంలో కరోనా వారియర్స్తో మాట్లాడుతూ ప్లాస్మా దానం చేయాలని అవగాహన కల్పిస్తున్నారు . ఇక సినిమాల విషయానికి వస్తే .. నాగచైతన్య , సాయిపల్లవి హీరోహీరోయిన్లుగా ‘లవ్స్టోరి’ సినిమాను తెరకెక్కిస్తున్నారు శేఖర్ కమ్ముల . ఈ సినిమా షూటింగ్ తుది దశకు చేరుకుంది . సినీ రంగానికి చెందిన పలువురు సోషల్ మీడియా ద్వారా సంతాపం వ్యక్తం చేశారు .

Top vertical assignments:
  1. movies  (p = 0.930)
  2. tv  (p = 0.019)
  3. entertainment  (p = 0

In [96]:
# Cell: Inspect Telugu article → vertical assignments
import numpy as np

def print_telugu_assignment(idx, top_k=5):
    """
    Print headline, article text, and top_k predicted verticals for the
    Telugu article at DataFrame index `idx`.
    """
    row = telugu_df.iloc[idx]
    print(f"Story ID : {row['story_id']}")
    print(f"Headline : {row['headline']}")
    print(f"Article  :\n{row['article']}\n")
    
    probs   = all_probs[idx]                     # shape (17,)
    top_idxs = np.argsort(-probs)[:top_k]        # indices of top_k verticals
    
    print("Top vertical assignments:")
    for rank, i in enumerate(top_idxs, start=1):
        print(f"  {rank}. {newcat[i]}  (p = {probs[i]:.3f})")
    print("\n" + "-"*80 + "\n")

# Example: inspect the first 5 articles
for idx in range(5):
    print_telugu_assignment(idx, top_k=5)


Story ID : 920427
Headline : శేఖర్ కమ్ముల తండ్రి శేషయ్య కన్నుమూత
Article  :
ప్రముఖ దర్శకులు శేఖర్ కమ్ముల ఇంట విషాదం నెలకొంది . ఆయన తండ్రి కమ్ముల శేషయ్య ( 89 ) కన్నుమూశారు . ఆయన గత కొద్దికాలంగా వృద్దాప్య సంబంధింత సమస్యలతో బాధపడుతున్నారు . ప్రస్తుత పరిస్థితుల్లో కొద్దికాలంగా ఇంటి వద్దే చికిత్స అందిస్తున్నారు . ఇటీవల ఆరోగ్యం క్షీణించడంతో స్థానిక ప్రైవేట్ హాస్పిటల్లో చేర్పించారు . అక్కడే చికిత్స పొందుతూ శనివారం ఉదయం 6 గంటలకు మరణించారు అని సన్నిహితులు తెలిపారు . శనివారం సాయంత్రం బన్సీలాల్ పేట స్మశాన వాటికలో అంత్యక్రియలు జరుగుతాయని కుటుంబ సభ్యులు తెలిపారు . శేఖర్ కమ్ముల లాక్డౌన్ సమయంలో కరోనా వారియర్స్తో మాట్లాడుతూ ప్లాస్మా దానం చేయాలని అవగాహన కల్పిస్తున్నారు . ఇక సినిమాల విషయానికి వస్తే .. నాగచైతన్య , సాయిపల్లవి హీరోహీరోయిన్లుగా ‘లవ్స్టోరి’ సినిమాను తెరకెక్కిస్తున్నారు శేఖర్ కమ్ముల . ఈ సినిమా షూటింగ్ తుది దశకు చేరుకుంది . సినీ రంగానికి చెందిన పలువురు సోషల్ మీడియా ద్వారా సంతాపం వ్యక్తం చేశారు .

Top vertical assignments:
  1. movies  (p = 0.930)
  2. tv  (p = 0.019)
  3. entertainment  (p = 0